# Auxiliar 11: Procesos de Decision de Markov y Q-Learning.
Professor: Shaharyar Kamal.
Assistant: Rodrigo Navarro M.

Contenidos:

1.  **Procesos de Decisión de Markov (MDP):** Cómo definir un problema de decisiones secuenciales.
2.  **Ecuaciones de Bellman:** Cómo definir la solución óptima de un MDP.
3.  **Programación Dinámica:** Cómo encontrar la solución óptima cuando se tiene el modelo completo del sistema.
4.  **Aprendizaje por Refuerzo (RL):** Qué hacer cuando no tenemos un modelo completo del sistema.
5.  **Q-Learning:** Ejemplo de algoritmo RL cercano a programación dinámica.

## 1. Procesos de Decisión de Markov (MDP)

Un MDP es un marco matemático para modelar la toma de decisiones en situaciones donde los resultados son en parte aleatorios y en parte están bajo el control de un **agente**.

La "Propiedad de Markov" es clave: **El futuro solo depende del presente, no del pasado.** Lo que suceda a continuación solo depende de tu estado actual y de la acción que tomes ahora.

Un MDP se define por una tupla de 5 elementos: $(S, A, P, R, \gamma)$

- **$S$ (States / Estados):** Un conjunto de todos los estados posibles en los que puede estar el agente. (Ej. *"En la casilla (0,1) del laberinto"*).
- **$A$ (Actions / Acciones):** Un conjunto de todas las acciones que el agente puede tomar. (Ej. *"Ir al Norte, Sur, Este, Oeste"*).
- **$P$ (Transition Model / Modelo de Transición):** La probabilidad de terminar en un estado $s'$ si tomamos la acción $a$ en el estado $s$. Se escribe $P(s' | s, a)$. (Ej. *"Si estoy en (0,1) y elijo 'Norte', hay un 80% de probabilidad de llegar a (0,2) y un 20% de que el viento me desvíe a (1,1)"*).
- **$R$ (Reward Function / Función de Recompensa):** La recompensa inmediata que recibe el agente después de una transición. Se escribe $R(s, a, s')$. (Ej. *"+10 por llegar al tesoro, -100 por caer en la lava, -0.1 por cada paso para incentivar la rapidez"*).
- **$\gamma$ (Gamma / Factor de Descuento):** Un número entre 0 y 1. Determina qué tan importantes son las recompensas futuras.
    - Si $\gamma=0$, el agente es "miope" y solo le importa la recompensa inmediata.
    - Si $\gamma=0.99$, el agente es "visionario" y valora mucho las recompensas futuras, incluso si están lejos.



El **objetivo** del agente es encontrar una **política** ($\pi$) —una estrategia que le diga qué acción tomar en cada estado— que **maximice la recompensa total acumulada y descontada**.

![MDP](https://i.sstatic.net/HQa9b.jpg)




---



## 2. Las Ecuaciones de Optimalidad de Bellman

Las ecuaciones de Bellman son una forma de pensar sobre este problema de forma recursiva. Descomponen el valor de estar en un estado en dos partes: **la recompensa inmediata** y **el valor (descontado) del estado al que llegas**.

Definimos dos funciones de "valor":

- **$V^*(s)$ (Value Function / Función de Valor):** ¿Cuál es la máxima recompensa futura que puedo esperar si empiezo en el estado $s$?
- **$Q^*(s, a)$ (Action-Value Function / Función de Valor-Acción):** ¿Cuál es la máxima recompensa futura que puedo esperar si empiezo en el estado $s$, tomo la acción $a$, y *luego* actúo de forma óptima?

### Ecuación de Optimalidad de Bellman para $V^*$

El valor óptimo de un estado $s$ es igual a la recompensa que obtendré al tomar **la mejor acción posible** $a$ desde ese estado, más el valor descontado del estado $s'$ al que esa acción me lleve (promediado por las probabilidades de transición).

$$V^*(s) = \max_{a} \sum_{s'} P(s' | s, a) \left[ R(s, a, s') + \gamma V^*(s')
\right]$$

### Ecuación de Optimalidad de Bellman para $Q^*$

Esta es la más importante para nosotros. El valor óptimo de tomar la acción $a$ en el estado $s$ es la recompensa inmediata que obtengo, más el valor descontado de estar en el siguiente estado $s'$, donde elegiré **la mejor acción posible $a'$** desde *ese* nuevo estado.

$$Q^*(s, a) = \sum_{s'} P(s' | s, a) \left[ R(s, a, s') + \gamma \max_{a'} Q^*(s', a')
\right]$$

**La intuición es clave:** El valor de una decisión hoy ($Q^*(s, a)$) se define por la recompensa inmediata más el valor de la *mejor decisión de mañana* ($\max_{a'} Q^*(s', a')$).

---

## 3. Relación con Programación Dinámica (PD)

Cuándo usar programación dinámica?

- La solución puede ser obtenida dividiendo el problema en sub-problemas.

- Los sub-problemas son recurrentes (así que sus soluciones pueden ser
reutilizadas).

Los MDPs cumplen lo anterior:

- Existen relaciones recursivas (ecuaciones de Bellman).

- Los valores que toman las funciones de valor pueden ser almacenados y
reutilizados.


Si conocemos **perfectamente** el MDP (es decir, conocemos todas las probabilidades $P(s' | s, a)$ y recompensas $R$), podemos resolver las ecuaciones de Bellman directamente usando Programación Dinámica.

Los algoritmos principales son:

- **Value Iteration (Iteración de Valor):** Empezamos con valores $V_0(s) = 0$ para todos los estados. Luego, aplicamos repetidamente la ecuación de Bellman como una regla de actualización hasta que los valores de V dejen de cambiar (converjan).

    $V_{k+1}(s) \leftarrow \max_{a} \sum_{s'} P(s' | s, a) \left[ R(s, a, s') + \gamma V_k(s') \right]$

- **Policy Iteration (Iteración de Política):** Un proceso de dos pasos donde 1) Evaluamos qué tan buena es una política actual (calculamos $V^\pi$) y 2) Mejoramos la política actuando "codiciosamente" (greedy) con respecto a esos valores. Repetimos hasta que la política sea estable.

**El gran problema:** En la mayoría de los problemas del mundo real (plantas industriales, robótica, finanzas), ¡**NO** conocemos el modelo $P$!

---

## 4. Aprendizaje por Refuerzo (RL)

RL entra en juego cuando **no tenemos el mapa** (el modelo $P$ y $R$). El agente debe aprender a tomar buenas decisiones interactuando con el entorno y observando los resultados (las recompensas).

![rl](https://miro.medium.com/1*NCOUSqJtdblFskOw-yHQsQ.png)

El agente aprende de la *experiencia*, que viene en forma de tuplas: $(s, a, r, s')$
(Estaba en el estado $s$, tomé la acción $a$, recibí la recompensa $r$, y terminé en el estado $s'$).

**Q-Learning es un algoritmo de RL "Model-Free" (Libre de Modelo):** No intenta aprender $P$ o $R$. En su lugar, aprende la función $Q^*(s, a)$ directamente a partir de las experiencias.


## 5. Ejemplo Q-Learning

Q-Learning aprende la función $Q^*$ iterativamente. Mantiene una tabla (la **Q-Table**) con una estimación del valor $Q(s, a)$ para cada par estado-acción.

Cuando el agente tiene una experiencia $(s, a, r, s')$, actualiza su estimación para $Q(s, a)$ usando esta regla de actualización, que es una versión " muestreada" de la ecuación de Bellman:

**Regla de Actualización de Q-Learning:**

$$Q(s, a) \leftarrow Q(s, a) + \alpha \left( \underbrace{r + \gamma \max_{a'} Q(s', a')}_{	\text{Objetivo}} - Q(s, a)
\right)$$

Donde:
- $\alpha$ (Alfa / Tasa de Aprendizaje): Qué tanto confiamos en la nueva información. (Qué tan grande es el paso que damos hacia el "Objetivo TD").
- $(r + \gamma \max_{a'} Q(s', a'))$: Es nuestra mejor *nueva estimación* del valor de $Q(s, a)$ basada en la experiencia. Se llama el **Objetivo de Diferencia Temporal (TD Target)**.
- El término en el paréntesis es el **Error de Diferencia Temporal (TD Error)**: la diferencia entre nuestra nueva estimación y la antigua.

### Problema: El Lago Congelado (Frozen Lake)

Vamos a implementar Q-Learning en el entorno clásico "Frozen Lake" de la librería `gymnasium`.

- **Entorno:** Una cuadrícula de 4x4.
- **Estados (S):** 16 casillas (0-15).
- **Acciones (A):** 4 (0=Izquierda, 1=Abajo, 2=Derecha, 3=Arriba).
- **Recompensas (R):** +1 por llegar a la meta (G), 0 en todos los demás estados.
- **Transiciones (P):** El hielo es resbaladizo (`is_slippery=True`). Si eliges una acción, hay 1/3 de probabilidad de ir en esa dirección, 1/3 de ir 90° a la izquierda y 1/3 de ir 90° a la derecha.

```
S F F F  (S: Start)
F H F H  (F: Frozen)
F F F H  (H: Hole)
H F F G  (G: Goal)
```

In [1]:
import gymnasium as gym
import numpy as np
import random
import time
from IPython.display import clear_output

### 5.2 Implementación

In [2]:
# Cargar el entorno
env = gym.make("FrozenLake-v1", is_slippery=True)

# Inicializacion de la Q-Table
num_states = env.observation_space.n
num_actions = env.action_space.n
q_table = np.zeros((num_states, num_actions))

# Definicion de Hiperparametros
num_episodes = 20000
max_steps_per_episode = 100

learning_rate = 0.1
discount_factor = 0.99

epsilon = 1.0
max_epsilon = 1.0
min_epsilon = 0.01
epsilon_decay_rate = 0.0005

# Entrenamiento de Q-Learning
print("Entrenando agente...")

for episode in range(num_episodes):
    # Reiniciar el entorno para un nuevo episodio
    state, info = env.reset()
    terminated = False
    truncated = False

    for step in range(max_steps_per_episode):
        # Exploracion vs Explotacion (Epsilon-Greedy)
        if random.uniform(0, 1) < epsilon:
            action = env.action_space.sample() # Explorar: tomar una accion aleatoria
        else:
            action = np.argmax(q_table[state, :]) # Explotar: tomar la mejor accion conocida

        # Dar un paso en el entorno
        new_state, reward, terminated, truncated, info = env.step(action)

        # Actualizar la Q-Table segun la Ecuación de Bellman

        # Q(s, a) <- Q(s, a) + α * [r + γ * max_a'(Q(s', a')) - Q(s, a)]
        old_value = q_table[state, action]
        next_max = np.max(q_table[new_state, :])

        td_target = reward + discount_factor * next_max
        td_error = td_target - old_value

        new_value = old_value + learning_rate * td_error
        q_table[state, action] = new_value

        # Actualizar el estado
        state = new_state

        # Si el juego terminó (cayo en un agujero o llego a la meta)
        if terminated or truncated:
            break

    # Decaimiento de Epsilon
    # Reducimos la exploracion a medida que el agente aprende
    epsilon = min_epsilon + (max_epsilon - min_epsilon) * np.exp(-epsilon_decay_rate * episode)

print("¡Entrenamiento finalizado!")
env.close()

Entrenando agente...
¡Entrenamiento finalizado!


### 5.3 Resultados: La Q-Table Aprendida
La tabla nos muestra el valor esperado para cada acción (Columnas: 0=Izq, 1=Abajo, 2=Der, 3=Arr) en cada estado (Filas: 0-15).

In [3]:
print("Q-Table Resultante:")
print(q_table)

# Extraer la politica (la mejor accion en cada estado)
policy = np.argmax(q_table, axis=1)
print("\nPolitica Extraida (0=Izq, 1=Abajo, 2=Der, 3=Arr):")
print(policy.reshape(4, 4))

Q-Table Resultante:
[[0.51516028 0.50129084 0.50159078 0.50262569]
 [0.39746993 0.39291465 0.3259231  0.48397482]
 [0.4205126  0.3909444  0.40941561 0.4616459 ]
 [0.2087767  0.32938386 0.29211607 0.44698247]
 [0.52926983 0.35576639 0.39024282 0.34482072]
 [0.         0.         0.         0.        ]
 [0.33931639 0.05864181 0.18677498 0.09144282]
 [0.         0.         0.         0.        ]
 [0.33259238 0.44809316 0.43532009 0.55933987]
 [0.54641929 0.61184777 0.47514203 0.37559137]
 [0.53735698 0.40372845 0.3603358  0.32413152]
 [0.         0.         0.         0.        ]
 [0.         0.         0.         0.        ]
 [0.39942107 0.64307272 0.72125974 0.4280733 ]
 [0.73345628 0.85200966 0.77867664 0.76028581]
 [0.         0.         0.         0.        ]]

Politica Extraida (0=Izq, 1=Abajo, 2=Der, 3=Arr):
[[0 3 3 3]
 [0 0 0 0]
 [3 1 0 0]
 [0 2 1 0]]


### 5.4 Prueba del Agente Entrenado
Ahora, veamos al agente jugar usando la política que aprendió (explotación pura, sin exploración).

In [4]:
eval_env = gym.make("FrozenLake-v1", is_slippery=True, render_mode="ansi")
num_eval_episodes = 5
sleep_time = 0.4

for episode in range(num_eval_episodes):
    # Reiniciar el entorno
    state, info = eval_env.reset()
    terminated = False
    truncated = False

    print(f"--- Episodio de Evaluación #{episode + 1} ---")
    time.sleep(1) # Pausa antes de empezar

    for step in range(max_steps_per_episode):
        # Limpiar la salida de la celda de Colab
        clear_output(wait=True)

        # Renderizar el estado actual y mostrarlo
        frame = eval_env.render()
        print(frame)
        print(f"Episodio: {episode + 1}, Paso: {step + 1}")

        # Tomar la mejor accion de la Q-Table
        action = np.argmax(q_table[state, :])

        # Dar el paso
        new_state, reward, terminated, truncated, info = eval_env.step(action)

        # Actualizar el estado
        state = new_state

        if terminated or truncated:
            clear_output(wait=True)
            frame = eval_env.render()
            print(frame)
            if reward == 1.0:
                print("¡Objetivo alcanzado!")
            else:
                print("¡Caíste en un agujero!")
            time.sleep(2)
            break

        time.sleep(sleep_time)

clear_output(wait=True)
eval_env.close()

  (Down)
SFFF
FHFH
FFFH
HFFG

¡Objetivo alcanzado!
